# Grilla de min_data_in_leaf sobre el dataset CON LAGS + Ranking Cero Fijo

En la comparación anterior usamos `min_data_in_leaf=1000` directamente (el ganador para B con lags **sin** rankear), sin volver a validarlo. El rankeo cambia la distribución de todas las variables (percentiles 1-100 en vez de escala absoluta), así que el óptimo podría no ser el mismo acá.

Esta notebook repite la grilla (10 semillas por candidato) sobre `competencia_01_con_lags_rankeado.csv.gz` -- el archivo que ya generó la notebook anterior -- para confirmar cuál `min_data_in_leaf` es realmente el mejor en la versión rankeada, antes de dar por cerrado ese hiperparámetro.

### Setup y carga del dataset YA rankeado (generado por la notebook anterior)

In [1]:
rm(list = ls())
gc()

require("data.table")
require("lightgbm")

PARAM_carpeta_datasets <- "C:/Users/Laura/Documents/DMEyF/datasets/"
PARAM_carpeta_exp      <- "C:/Users/Laura/Documents/DMEyF/exp/"

dataset <- fread(paste0(PARAM_carpeta_datasets, "competencia_01_con_lags_rankeado.csv.gz"), stringsAsFactors = TRUE)
if (!("clase_ternaria" %in% colnames(dataset))) stop("No se encontro clase_ternaria en competencia_01_con_lags_rankeado.csv.gz")

dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]
campos_buenos <- setdiff(colnames(dataset), c("clase_ternaria", "clase01", "numero_de_cliente"))
campos_buenos <- setdiff(campos_buenos, c("cprestamos_personales", "mprestamos_personales"))

dataset_train <- dataset[foto_mes == 202104]
dfuture       <- dataset[foto_mes == 202106]

dtrain <- lgb.Dataset(
  data = data.matrix(dataset_train[, campos_buenos, with = FALSE]),
  label = dataset_train[, clase01]
)

cat(length(campos_buenos), "variables,", nrow(dataset_train), "filas train\n")

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,653260,34.9,1447557,77.4,1109001,59.3
Vcells,1199500,9.2,8388608,64.0,1928501,14.8


Caricamento del pacchetto richiesto: data.table

Caricamento del pacchetto richiesto: lightgbm



751 variables, 163284 filas train


### Hiperparámetros base y grilla

In [2]:
param_B_base <- list(
  boosting= "gbdt", objective= "binary", metric= "auc",
  first_metric_only= FALSE, boost_from_average= TRUE, feature_pre_filter= FALSE,
  force_row_wise= TRUE, verbosity= -100,
  max_depth= -1L, max_bin= 31L,
  min_sum_hessian_in_leaf = 0.001,
  pos_bagging_fraction= 1.0, neg_bagging_fraction= 1.0, is_unbalance= FALSE,
  drop_rate= 0.1, max_drop= 50, skip_drop= 0.5, extra_trees= FALSE,
  num_iterations = 600, learning_rate = 0.02, feature_fraction = 0.1541961,
  num_leaves = 1684, scale_pos_weight = 0.5,
  min_gain_to_split = 0.2158453, lambda_l1 = 1.520327, lambda_l2 = 7.548036,
  bagging_fraction = 0.8425105, bagging_freq = 6
)

CANDIDATOS_MIN_DATA <- c(460, 1000, 2000, 3500, 5000)
CORTE_FIJO <- 12000

SEMILLAS_10 <- c(585581, 249449, 655649, 503039, 798191,
                  112909, 334667, 421303, 667021, 887923)

### Loop: por min_data_in_leaf, por semilla (con checkpoint por si se corta)

In [3]:
archivo_checkpoint <- paste0(PARAM_carpeta_exp, "grilla_rankeado_lags.txt")

ya_hechas <- data.table(min_data_in_leaf = integer(), semilla = integer())
if (file.exists(archivo_checkpoint)) {
  ya_hechas <- fread(archivo_checkpoint)[, list(min_data_in_leaf, semilla)]
}

for (md in CANDIDATOS_MIN_DATA) {
  for (SEM in SEMILLAS_10) {

    if (nrow(ya_hechas[min_data_in_leaf == md & semilla == SEM]) > 0) next

    pB <- modifyList(param_B_base, list(min_data_in_leaf = md, seed = SEM))

    set.seed(SEM, "L'Ecuyer-CMRG")
    modelo <- lgb.train(data = dtrain, param = pB)

    pred <- predict(modelo, data.matrix(dfuture[, campos_buenos, with = FALSE]))
    tb <- dfuture[, list(clase_ternaria)]
    tb[, prob := pred]
    setorder(tb, -prob)
    tb[, Predicted := 0L]
    tb[1:CORTE_FIJO, Predicted := 1L]
    gan <- tb[Predicted == 1L, sum(ifelse(clase_ternaria == "BAJA+2", 1072500, -27500))]

    cat("min_data_in_leaf=", md, " semilla=", SEM, " ganancia=", gan, "\n", sep = "")

    fwrite(
      data.table(min_data_in_leaf = md, semilla = SEM, ganancia = gan),
      file = archivo_checkpoint, sep = "\t", append = file.exists(archivo_checkpoint)
    )

    rm(modelo)
    gc(full = TRUE, verbose = FALSE)
  }
}

min_data_in_leaf=460 semilla=585581 ganancia=456500000
min_data_in_leaf=460 semilla=249449 ganancia=465300000
min_data_in_leaf=460 semilla=655649 ganancia=470800000
min_data_in_leaf=460 semilla=503039 ganancia=457600000
min_data_in_leaf=460 semilla=798191 ganancia=475200000
min_data_in_leaf=460 semilla=112909 ganancia=469700000
min_data_in_leaf=460 semilla=334667 ganancia=469700000
min_data_in_leaf=460 semilla=421303 ganancia=465300000
min_data_in_leaf=460 semilla=667021 ganancia=4.73e+08
min_data_in_leaf=460 semilla=887923 ganancia=4.73e+08
min_data_in_leaf=1000 semilla=585581 ganancia=460900000
min_data_in_leaf=1000 semilla=249449 ganancia=467500000
min_data_in_leaf=1000 semilla=655649 ganancia=471900000
min_data_in_leaf=1000 semilla=503039 ganancia=457600000
min_data_in_leaf=1000 semilla=798191 ganancia=471900000
min_data_in_leaf=1000 semilla=112909 ganancia=467500000
min_data_in_leaf=1000 semilla=334667 ganancia=470800000
min_data_in_leaf=1000 semilla=421303 ganancia=465300000
min_

### Resumen

In [4]:
resultados <- fread(archivo_checkpoint)
resumen <- resultados[, .(ganancia_media = mean(ganancia), ganancia_desvio = sd(ganancia)), by = min_data_in_leaf]
setorder(resumen, min_data_in_leaf)
print(resumen)

cat("\n--- Referencias ---\n")
cat("B con lags, sin rankear (min_data=1000):        Media=464090000  Desvio=4097818\n")
cat("B con lags, con rankeo, min_data=1000 (previo):  Media=468050000  Desvio=5493885\n")

   min_data_in_leaf ganancia_media ganancia_desvio
              <int>          <num>           <num>
1:              460      467610000         6402508
2:             1000      468050000         5493885
3:             2000      465850000         2804461
4:             3500      462440000         2212188
5:             5000      451770000         4018305

--- Referencias ---
B con lags, sin rankear (min_data=1000):        Media=464090000  Desvio=4097818
B con lags, con rankeo, min_data=1000 (previo):  Media=468050000  Desvio=5493885


### Cómo seguir

Si algún `min_data_in_leaf` distinto de 1000 da una media más alta (o similar con menor desvío) sobre el dataset rankeado, ese pasa a ser el candidato para el modelo final con rankeo -- no asumas que 1000 sigue siendo el mejor solo porque lo era para la versión sin rankear.